In [6]:
import yaml
from pathlib import Path

import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftModelForCausalLM, PeftMixedModel

from src.training.ppo_trainer import train_ppo
from src.models.loader import load_model_and_tokenizer, load_value_model, load_model_from_checkpoint
from src.utils.data_utils import prepare_ppo_dataset_with_gt
from src.rewards.sparse import sparse_reward
from src.rewards.reward_module import RuleBasedRewardWrapper
from src.training.custom_ppo_trainer import GroundTruthPPOTrainer
from src.utils.data_utils import extract_gsm8k_answer, load_gsm8k, format_chat_prompt, extract_gsm8k_reasoning
from src.utils.math_verify import extract_answer_from_response, extract_thinking_from_response, verify_response
from src.evaluation.evaluate import generate_responses_batched
from src.models.loader import *


def load_config(config_path: str = None) -> dict:
    """Load and merge base + experiment configs."""
    base_path = Path("configs/base.yaml")
    with open(base_path) as f:
        cfg = yaml.safe_load(f)

    if config_path:
        with open(config_path) as f:
            exp_cfg = yaml.safe_load(f)
        # Shallow merge: experiment config overrides base
        for key, value in exp_cfg.items():
            if isinstance(value, dict) and key in cfg and isinstance(cfg[key], dict):
                cfg[key].update(value)
            else:
                cfg[key] = value

    return cfg

In [12]:
import yaml
with open("configs/base.yaml") as f:
    cfg = yaml.safe_load(f)

from src.utils.seed import set_seed
set_seed(cfg.get("seed", 42))

from src.models.loader import load_model_from_checkpoint
model, tokenizer = load_model_from_checkpoint("results/ppo_sparse_pretrained/checkpoint-200", cfg)
model = load_adapters(model, "results/ppo_sparse_pretrained/checkpoint-200", ["ppo_adapter"])
tokenizer.padding_side = "left"

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# lora_config = get_lora_config(cfg)
# model.add_adapter("ppo_adapter", lora_config)
# model.base_model.set_adapter(["default", "ppo_adapter"])

# for name, param in model.named_parameters():
#     if "lora_" in name and ".default." in name:
#         param.requires_grad = False

# model.print_trainable_parameters()

trainable params: 35,192,832 || all params: 564,418,432 || trainable%: 6.2352


Reproduction of the code in evaluate.py

In [13]:
ds = load_gsm8k("test")

questions = [ex["question"] for ex in ds]
ground_truths = [extract_gsm8k_answer(ex["answer"]) for ex in ds]
prompts = [format_chat_prompt(q, tokenizer) for q in questions]


model.eval()
responses = generate_responses_batched(model, tokenizer, prompts[:64], 512, batch_size=128)

results = []
for question, gt, response in zip(prompts, ground_truths, responses):
    result = {}
    result["verification"] = verify_response(response, gt)
    result["question"] = question
    result["ground_truth"] = gt
    result["response"] = response
    results.append(result.copy())

print(results[0]["response"])

# Aggregate metrics
n = len(results)
metrics = {
    "n_examples": n,
    "accuracy": sum(r["verification"]["correct"] for r in results) / n,
    "format_adherence": sum(r["verification"]["has_format"] for r in results) / n,
    # "accuracy+format_adherence": sum(r["correct"] for r in results if r["has_format"]) / sum(r["has_format"] for r in results),
    "has_reasoning": sum(r["verification"]["has_reasoning"] for r in results) / n,
    "answer_extraction_rate": sum(
        r["verification"]["predicted_answer"] is not None for r in results
    ) / n,
    "avg_reasoning_length": (
        sum(r["verification"]["reasoning_length"] for r in results) / n
    ),
}
metrics

Generating responses for 64 examples (batch_size=128)...: 100%|██████████| 1/1 [00:22<00:00, 22.16s/it]

<think>
First find how many eggs the ducks lay each day: 16 eggs/day + 3 eggs = 19 eggs/day
Then subtract the number of eggs the family buys from the store to find the net profit: 19 eggs/day - 4 eggs/day = 15 eggs/day
</think>
<answer>15</answer>


{'n_examples': 64,
 'accuracy': 0.0625,
 'format_adherence': 1.0,
 'has_reasoning': 1.0,
 'answer_extraction_rate': 1.0,
 'avg_reasoning_length': 34.265625}